In [ ]:
#| default_exp features.wps_panel

In [ ]:
#| export
from __future__ import annotations
import pandas as pd
import numpy as np
import structlog
from kreview.eval_engine import FeatureEvaluator

log = structlog.get_logger()
        

In [ ]:
#| export
def _parse_array(s):
    if not isinstance(s, str) or not s.startswith('['): 
        return []
    clean = s.replace('[', '').replace(']', '').replace(chr(10), '').replace(chr(13), '')
    try:
        return [float(x) for x in clean.split()]
    except (ValueError, TypeError):
        return []

class WPSPanelEvaluator(FeatureEvaluator):
    """Extracts WPS nucleosome binding geometries."""
    
    name = "WPSPanel"
    source_file = ".WPS.panel.parquet"
    tier = 2
    category = "epigenetics_and_geometry"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        extracted = {}
        try:
            if df.empty:
                return extracted
            cols = set(df.columns)

            if "region_type" in cols:
                array_cols = ["wps_nuc", "wps_tf", "prot_frac_nuc", "prot_frac_tf"]
                float_cols = ["local_depth"]
                for row in df.to_dict('records'):
                    rt = str(row["region_type"]).replace(" ", "_").replace("-","_").replace("|","_")
                    
                    for m in float_cols:
                        if m in cols and pd.notna(row[m]):
                            extracted[f"{rt}_{m}"] = float(row[m])
                            
                    for a in array_cols:
                        if a in cols and pd.notna(row[a]):
                            parsed = _parse_array(str(row[a]))
                            if len(parsed) > 0:
                                extracted[f"{rt}_{a}_mean"] = float(np.mean(parsed))
                                extracted[f"{rt}_{a}_std"] = float(np.std(parsed))
    
            return extracted
        except Exception as e:
            log.exception("extraction_failed", evaluator=self.name, error=str(e))
            return {}


In [ ]:
#| test
from kreview.features.wps_panel import _parse_array
evaluator = WPSPanelEvaluator()
assert len(_parse_array("[0.1 0.2]")) == 2
df = pd.DataFrame([{"region_type": "CTCF", "wps_nuc": "[1.0 2.0 3.0]", "local_depth": 0.5}])
res = evaluator.extract(df)
assert res["CTCF_wps_nuc_mean"] == 2.0
assert res["CTCF_local_depth"] == 0.5
